# Calculating power loss in Switching Transistors & MOSFETs

In [1]:
import matplotlib.pyplot as plt
import numpy as np

## Transistors and IGBTs

- $P_{SW} = \frac{1}{6}V_{CE}(max) \cdot I_{C}(max)$
- $P_{on} = I_C(max) \cdot V_{CE}(sat)$
- $P_{off} = I_C(leak) \cdot V_{CE}(max)$

$ P_{total} = (P_{SW} \cdot {t_{sw}}/{T}) + (P_{on} \cdot {t_{on}}/{T}) + (P_{off} \cdot {t_{off}}/{T})$

Note:
- $t_{sw} = t_{rise} + t_{fall}$

In [2]:
def Psw_tr_f(vce_max, ic_max):
    return (1 / 6) * vce_max * ic_max


def Pon_tr_f(ic_max, vce_sat):
    return ic_max * vce_sat


def Poff_tr_f(ic_leak, vce_max):
    return ic_leak * vce_max


def Ptotal_transistor_f(ton, toff, T, vce_max, ic_max, vce_sat, ic_leak, tsw):
    return (
        (Psw_tr_f(vce_max, ic_max) * (tsw / T))
        + (Pon_tr_f(ic_max, vce_sat) * (ton / T))
        + (Poff_tr_f(ic_leak, vce_max) * (toff / T))
    )

In [3]:
def summary_tr(f, D, Vce_max, Ic_max, Vce_sat, Ic_leak, tsw):
    T = 1 / f
    ton = (D / 100) * T
    toff = (1 - (D / 100)) * T

    print(f"Duty Cycle = {D}%, Switching Frequency = {f:0.3e} Hz, Period = {T:0.3e} s")
    print("Testing if tsw << T:", end=" ")
    if T > tsw:
        print(f"Yes, T = {T:0.3e} s, T/tsw = {T / tsw:.3e}\n")
        print(f"Pon = {Pon_tr_f(Ic_max, Vce_sat):0.3e} W, on time = {ton:0.3e} s")
        print(f"Poff = {Poff_tr_f(Ic_leak, Vce_max):0.3e} W, off time = {toff:0.3e} s")
        print(
            f"Psw = {Psw_tr_f(Vce_max, Ic_max):0.3e} W, switching time = {tsw:0.3e} s"
        )
        print(
            f"Total Power = {Ptotal_transistor_f(ton, toff, T, Vce_max, Ic_max, Vce_sat, Ic_leak, tsw):0.4f} W, "
            f"{Ptotal_transistor_f(ton, toff, T, Vce_max, Ic_max, Vce_sat, Ic_leak, tsw):0.4e} W"
        )

    else:
        print(f"No, T = {T:0.3e} s")
        print(f"T/tsw = {T / tsw:.3e}\n")

    print("-------------------------------------------------")


In [4]:
# Transistor/IGBT specs

Vce_max = 5  # max voltage (V)
Vce_sat = 0.2  # saturation voltage (V)

Ic_max = 1.4  # max current (amps)
Ic_leak = 100e-9  # leakage current (amps)

trise = 2.5e-9  # rise time (seconds)
tfall = 4e-9  # fall time (seconds)
tsw = trise + tfall  # switching time

In [5]:
# summary_tr(
#     f=1e3,
#     D=100,
#     Vce_max=Vce_max,
#     Ic_max=Ic_max,
#     Vce_sat=Vce_sat,
#     Ic_leak=Ic_leak,
#     tsw=tsw,
# )
# summary_tr(
#     f=1e3,
#     D=10,
#     Vce_max=Vce_max,
#     Ic_max=Ic_max,
#     Vce_sat=Vce_sat,
#     Ic_leak=Ic_leak,
#     tsw=tsw,
# )

<hr>

## MOSFETs

- $P_{on} = I_D^2 \cdot R_{DS}(on) $
- $P_{off} = I_{DSS} \cdot V_{DS}(max) $
- $P_{sw} = \frac{1}{6}V_{DS}(max) \cdot I_D $

$ P_{total} = (P_{SW} \cdot \frac{t_{sw}}{T}) + (P_{on} \cdot \frac{t_{on}}{T}) + (P_{off} \cdot \frac{t_{off}}{T})$

Note:
- $t_{sw} = t_{rise} + t_{fall}$

In [6]:
def Pon_mos_f(I_D, R_DS_ON):
    return I_D**2 * R_DS_ON


def Poff_mos_f(I_DSS, V_DS):
    return I_DSS * V_DS


def Psw_mos_f(V_DS, I_D):
    return (1 / 6) * V_DS * I_D

def Ptotal_mos_f(ton, toff, T, V_DS, I_D, R_DS_ON, I_DSS, tsw):
    return (
        (Psw_mos_f(V_DS, I_D) * (tsw / T))
        + (Pon_mos_f(I_D, R_DS_ON) * (ton / T))
        + (Poff_mos_f(I_DSS, V_DS) * (toff / T))
    )

In [7]:
def summary_mos(f, D, V_DS, I_D, R_DS_ON, I_DSS, tsw):
    T = 1 / f
    ton = (D / 100) * T
    toff = (1 - (D / 100)) * T

    print(f"Switching Frequency = {f:0.3e} Hz, Period = {T:0.3e} s, Duty Cycle = {D}%, R_DS_ON = {R_DS_ON:0.3e} ohms")
    print("Testing if tsw << T:", end=" ")
    if T > tsw:
        print(f"Yes, T = {T:0.3e} s, T/tsw = {T / tsw:.3e}\n")
        print(f"Pon = {Pon_mos_f(I_D, R_DS_ON):0.3e} W, on time = {ton:0.3e} s")
        print(f"Poff = {Poff_mos_f(I_DSS, V_DS):0.3e} W, off time = {toff:0.3e} s")
        print(f"Psw = {Psw_mos_f(V_DS, I_D):0.3e} W, switching time = {tsw:0.3e} s")
        print(
            f"Total Power = {Ptotal_mos_f(ton, toff, T, V_DS, I_D, R_DS_ON, I_DSS, tsw):0.4f} W | "
            f"{Ptotal_mos_f(ton, toff, T, V_DS, I_D, R_DS_ON, I_DSS, tsw):0.4e} W"
        )

    else:
        print(f"No, T = {T:0.3e} s")
        print(f"T/tsw = {T / tsw:.3e}\n")

    print("-------------------------------------------------")

In [8]:
# Mosfet specs for A03400A (https://mm.digikey.com/Volume0/opasdata/d220001/medias/docus/1133/AO3400A_ds.pdf)

I_D = 1.4  # drain current of the MOSFET
V_DS = 5  # drain-source voltage of the MOSFET
R_DS_ON = 48e-3  # internal resistance of the MOSFET
I_DSS = 5e-6  # leakage current of the MOSFET

ton = 2.5e-9  # turn-on time (seconds)
toff = 4e-9  # turn-off time (seconds)
tsw = trise + tfall

In [9]:
summary_mos(f=1e3, D=100, V_DS=V_DS, I_D=I_D, R_DS_ON=R_DS_ON, I_DSS=I_DSS, tsw=tsw)
summary_mos(f=1e3, D=100, V_DS=V_DS, I_D=I_D, R_DS_ON=32e-3, I_DSS=I_DSS, tsw=tsw)

Switching Frequency = 1.000e+03 Hz, Period = 1.000e-03 s, Duty Cycle = 100%, R_DS_ON = 4.800e-02 ohms
Testing if tsw << T: Yes, T = 1.000e-03 s, T/tsw = 1.538e+05

Pon = 9.408e-02 W, on time = 1.000e-03 s
Poff = 2.500e-05 W, off time = 0.000e+00 s
Psw = 1.167e+00 W, switching time = 6.500e-09 s
Total Power = 0.0941 W | 9.4088e-02 W
-------------------------------------------------
Switching Frequency = 1.000e+03 Hz, Period = 1.000e-03 s, Duty Cycle = 100%, R_DS_ON = 3.200e-02 ohms
Testing if tsw << T: Yes, T = 1.000e-03 s, T/tsw = 1.538e+05

Pon = 6.272e-02 W, on time = 1.000e-03 s
Poff = 2.500e-05 W, off time = 0.000e+00 s
Psw = 1.167e+00 W, switching time = 6.500e-09 s
Total Power = 0.0627 W | 6.2728e-02 W
-------------------------------------------------
